# Lecture 3 — Class Exercise
## Line Charts: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
- No spaghetti — if you show multiple lines, grey everything and highlight one
- Remove chart borders, heavy gridlines, rotating text
- Label directly; avoid legends where possible
- Insight title


In [8]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [9]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


## Task 1 - A line chart showing CO2 per Capita for all countries in the dataset

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#EEEEEE), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

In [10]:
# Task 1: CO2 per capita for all countries, highlighting Germany

highlight_country = "Germany"

fig = go.Figure()

# All other countries provide context in thin, light grey lines.
for country, group in df.groupby("Country"):
    if country == highlight_country:
        continue
    group = group.sort_values("Year")
    fig.add_trace(go.Scatter(
        x=group["Year"],
        y=group["CO2_per_capita"],
        mode="lines",
        line=dict(color="#EEEEEE", width=1),
        hovertemplate=(
            f"{country}<br>Year: %{{x}}"
            "<br>CO2 per capita: %{y:.2f}<extra></extra>"
        ),
        showlegend=False
    ))

# Highlight Germany and label the final point directly.
highlight = df[df["Country"] == highlight_country].sort_values("Year")
if highlight.empty:
    raise ValueError(f"{highlight_country} is not present in the dataset.")

fig.add_trace(go.Scatter(
    x=highlight["Year"],
    y=highlight["CO2_per_capita"],
    mode="lines+markers",
    line=dict(color="#D62728", width=3),
    marker=dict(size=5),
    hovertemplate=(
        "Germany<br>Year: %{x}"
        "<br>CO2 per capita: %{y:.2f}<extra></extra>"
    ),
    showlegend=False
))

last = highlight.iloc[-1]
fig.add_annotation(
    x=last["Year"], y=last["CO2_per_capita"], text=highlight_country,
    showarrow=False, xanchor="left", xshift=7,
    font=dict(color="#D62728", size=12)
)

fig.update_layout(
    title=dict(
        text="Germany's CO2 per-capita emissions have declined since 2000",
        x=0.02, xanchor="left"
    ),
    xaxis_title="Year",
    yaxis_title="CO2 per capita (dataset units)",
    template="plotly_white",
    plot_bgcolor="white",
    paper_bgcolor="white",
    showlegend=False,
    margin=dict(l=60, r=100, t=85, b=55)
)
fig.update_xaxes(showgrid=False, showline=False, zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor="#EEEEEE", showline=False, zeroline=False)
fig.show()


---
## Task 2 — A line chart showing CO2 emissions over time for all Asian countries in the dataset, with one country highlighted.

**same requirements as Task1**

In [11]:
# Task — Multi-series line with highlight
# YOUR CODE HERE
# Task 2: CO2 emissions over time for Asian countries, highlighting China

asia_df = df[df["Region"].eq("Asia")].copy()
highlight_country = "China"

if asia_df.empty:
    raise ValueError("No rows found for Region == 'Asia'. Check the dataset.")
if highlight_country not in asia_df["Country"].unique():
    raise ValueError(f"{highlight_country} is not present among Asian countries.")

fig = go.Figure()

# All other Asian countries provide context in thin, light grey lines.
for country, group in asia_df.groupby("Country"):
    if country == highlight_country:
        continue
    group = group.sort_values("Year")
    fig.add_trace(go.Scatter(
        x=group["Year"],
        y=group["CO2_Mt"],
        mode="lines",
        line=dict(color="#EEEEEE", width=1),
        hovertemplate=(
            f"{country}<br>Year: %{{x}}"
            "<br>CO2 emissions: %{y:.1f} Mt<extra></extra>"
        ),
        showlegend=False
    ))

# Highlight China and label its final point directly.
highlight = asia_df[asia_df["Country"] == highlight_country].sort_values("Year")
fig.add_trace(go.Scatter(
    x=highlight["Year"],
    y=highlight["CO2_Mt"],
    mode="lines+markers",
    line=dict(color="#D62728", width=3),
    marker=dict(size=5),
    hovertemplate=(
        "China<br>Year: %{x}"
        "<br>CO2 emissions: %{y:.1f} Mt<extra></extra>"
    ),
    showlegend=False
))

last = highlight.iloc[-1]
fig.add_annotation(
    x=last["Year"], y=last["CO2_Mt"], text=highlight_country,
    showarrow=False, xanchor="left", xshift=7,
    font=dict(color="#D62728", size=12)
)

fig.update_layout(
    title=dict(
        text="China's CO2 emissions have risen sharply since 2000",
        x=0.02, xanchor="left"
    ),
    xaxis_title="Year",
    yaxis_title="CO2 emissions (Mt)",
    template="plotly_white",
    plot_bgcolor="white",
    paper_bgcolor="white",
    showlegend=False,
    margin=dict(l=60, r=100, t=85, b=55)
)
fig.update_xaxes(showgrid=False, showline=False, zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor="#EEEEEE", showline=False, zeroline=False)
fig.show()
